# Energy-optimal speed profile by dynamic programming

**Goal.** Drive a small electric vehicle over a fixed distance within a time budget while spending as little electrical energy as possible. At every position step the controller chooses a motor current.

**Method.** Finite-horizon dynamic programming (backward induction on the Bellman equation) over the state *(speed, position, remaining time)*. The vehicle model is known, so this is *model-based planning*, not model-free reinforcement learning: nothing is learned by trial and error, and the optimum on the grid is computed exactly.

**Status.** Simulation only. The model coefficients `K1`-`K4` are generic placeholders that will be identified on the real vehicle (work in progress).

> A complete run takes about 20 minutes on an Apple M1 and needs roughly 4 GiB of RAM. The results of that run are shown and discussed in [`README.md`](README.md).


## 1. Model

One decision is taken per **position step** `D_POS` (the controller is position-triggered, not clock-triggered). The action is the motor current `i`.

| Quantity | Formula |
|---|---|
| Speed after one step | `v' = sqrt(K1·v² + K2·i)` |
| Electrical power | `P = K3·v'·i + K4·i²` (evaluated at `v'`) |
| Step duration | `Δt = D_POS / v'` |
| Step energy | `E = P·Δt = P·D_POS / v'` |

The step is assumed to be travelled at the speed reached at its end, `v'`. The time budget is enforced exactly by keeping the **remaining time in the state**, so no Lagrange multiplier is needed.

Bellman recursion, with `V = 0` at the goal and `V = +inf` when time runs out before the goal is reached:

`V(v, x, t) = min over i of [ E(v, i) + V(v', x + D_POS, t - Δt(v, i)) ]`

**Index convention.** All tables are indexed by grid *indices*; the physical value is the index times the grid spacing (for example `speed = v_idx * D_SPEED`).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 2. Parameters and grids

| Variable | Points | Spacing | Range |
|---|---|---|---|
| Motor current | `N_CURRENT = 600` | 0.004 A | 0 - 2.4 A |
| Speed | `N_SPEED = 2000` | 0.0005 m/s | 0 - 1.0 m/s |
| Position | `N_POS = 300` | 5 mm | 299 steps = 1.495 m |
| Time | `N_TIME = 450` | 0.05 s | 0 - 22.5 s |


In [ ]:
# --- Grid resolution (number of discrete values per variable) ---
N_CURRENT = 600   # motor current levels
N_SPEED = 2000    # speed levels
N_POS = 300       # position points
N_TIME = 450      # time bins

# --- Grid spacing (physical units) ---
D_SPEED = np.float64(0.0005)    # m/s per speed index
D_CURRENT = np.float64(0.004)   # A per current index
D_POS = np.float64(0.005)       # m per position step
D_TIME = np.float64(0.05)       # s per time bin

# --- Vehicle model coefficients (generic placeholders, to be identified on hardware) ---
K1 = np.float64(0.5665)   # fraction of v^2 kept after one step (< 1: losses)
K2 = np.float64(0.018)    # gain in v^2 per ampere
K3 = np.float64(0.4)      # coefficient of the v*i power term
K4 = np.float64(0.6)      # coefficient of the i^2 power term (resistive-like loss)


In [ ]:
# Physical value of each grid index. Kept as in the original code: the endpoint
# spacing is 1/1999 rather than exactly 1/2000, which is negligible here.
speed_grid = np.linspace(0, N_SPEED * D_SPEED, N_SPEED)   # m/s
time_grid = np.linspace(0, N_TIME * D_TIME, N_TIME)       # s

# 2-D index grids over every (speed, current) pair, shape (N_SPEED, N_CURRENT)
I_IDX, V_IDX = np.meshgrid(np.arange(N_CURRENT), np.arange(N_SPEED))


## 3. One-step model

Each function takes a speed index and a current index (scalars or arrays) and returns the corresponding quantity for one position step.

In [ ]:
def power(v_idx, i_idx):
    """Electrical power [W]: P = K3*v*i + K4*i^2."""
    v = v_idx * D_SPEED
    i = i_idx * D_CURRENT
    return K3 * v * i + K4 * i**2


def next_speed_index(v_idx, i_idx):
    """Speed index reached after one position step: v' = sqrt(K1*v^2 + K2*i),
    rounded down onto the speed grid."""
    v_next = (K1 * (v_idx * D_SPEED) ** 2 + K2 * (i_idx * D_CURRENT)) ** 0.5
    return np.digitize(v_next, speed_grid) - 1


def step_duration_bins(v_idx, i_idx):
    """Number of time bins needed to cross one position step at the speed v'
    reached at its end (duration = D_POS / v'). Standing still (v' = 0) gives
    the largest value, N_TIME."""
    duration = D_POS / (D_SPEED * next_speed_index(v_idx, i_idx))
    return np.digitize(duration, time_grid)


def step_energy(v_idx, i_idx):
    """Energy [J] spent on one position step: P(v', i) * D_POS / v'.
    The tiny constant keeps the numerator strictly positive, so standing still
    (v' = 0) costs +inf instead of NaN (kept from the original implementation)."""
    v_next_idx = next_speed_index(v_idx, i_idx)
    return (power(v_next_idx, i_idx) * D_POS + power(0, 1) / 100000) / (v_next_idx * D_SPEED)


### Pre-computed tables

For every (speed, current) pair: energy per step, duration in time bins, and next speed. Infinite values at `v' = 0` are intentional (standing still never reaches the goal), hence the silenced division warning.

In [ ]:
with np.errstate(divide="ignore"):
    STEP_ENERGY = step_energy(V_IDX, I_IDX)            # [J]
    STEP_DURATION = step_duration_bins(V_IDX, I_IDX)   # [time bins]
NEXT_SPEED = next_speed_index(V_IDX, I_IDX)            # [speed index]


**Step energy at constant current.** Four currents are shown (0, 0.4, 0.8 and 1.196 A).

In [ ]:
for i_idx in (299, 200, 100, 0):
    plt.plot(STEP_ENERGY[:, i_idx], marker="o", label=f"i = {i_idx * D_CURRENT:.2f} A")
plt.xlabel("Speed index (1 index = 0.5 mm/s)")
plt.ylabel("Energy per 5 mm step [J]")
plt.title("Step energy vs. speed, at constant current")
plt.legend()
plt.show()


**Step duration at constant current** (log scale: standing still takes the maximum, `N_TIME` bins).

In [ ]:
for i_idx in (299, 200, 100, 0):
    plt.plot(STEP_DURATION[:, i_idx], marker="o", label=f"i = {i_idx * D_CURRENT:.2f} A")
plt.yscale("log")
plt.xlabel("Speed index (1 index = 0.5 mm/s)")
plt.ylabel("Step duration [time bins of 0.05 s]")
plt.title("Step duration vs. speed, at constant current")
plt.legend()
plt.show()


**Speed after one step** versus speed before the step, at constant current.

In [ ]:
for i_idx in (299, 200, 100, 0):
    plt.plot(NEXT_SPEED[:, i_idx], marker="o", label=f"i = {i_idx * D_CURRENT:.2f} A")
plt.xlabel("Speed index before the step (1 index = 0.5 mm/s)")
plt.ylabel("Speed index after the step")
plt.title("Speed update over one position step")
plt.legend()
plt.show()


## 4. Dynamic programming (backward induction)

`min_energy[v, x, t]` is the minimum energy needed to reach the goal from speed index `v` at position index `x` with `t` time bins left. `best_current[v, x, t]` is the current index that achieves it. The table is filled from the goal backwards.

Memory: both tables have `N_SPEED * N_POS * N_TIME` = 270 million entries (about 4 GiB in total).

In [ ]:
def solve_dp():
    """Fill the value table by backward induction.

    Returns
    -------
    best_current : int array (N_SPEED, N_POS - 1, N_TIME)
        Optimal current index for each (speed, position, time left).
    min_energy : float array (N_SPEED, N_POS, N_TIME)
        Minimum energy [J] to reach the goal from each (speed, position, time left).
    """
    best_current = np.zeros((N_SPEED, N_POS - 1, N_TIME), dtype=int)
    min_energy = np.zeros((N_SPEED, N_POS, N_TIME))
    min_energy[:, :-1, 0] = np.inf   # no time left and goal not reached: infeasible

    for steps_from_goal in range(1, N_POS):
        pos = N_POS - 1 - steps_from_goal   # position index being filled
        next_pos = pos + 1                  # position index after one step
        for t in range(1, N_TIME):
            time_left = t - STEP_DURATION   # time bins left after the step, shape (N_SPEED, N_CURRENT)
            feasible = time_left >= 0       # the step must fit in the remaining time
            step_cost = np.where(feasible, STEP_ENERGY, np.inf)
            # Negative indices wrap around, but np.where discards those entries.
            future_cost = np.where(feasible, min_energy[NEXT_SPEED, next_pos, time_left], np.inf)
            total = step_cost + future_cost
            min_energy[:, pos, t] = total.min(axis=1)       # best over all currents
            best_current[:, pos, t] = total.argmin(axis=1)  # current that achieves it
    return best_current, min_energy


In [ ]:
best_current, min_energy = solve_dp()   # about 20 minutes on an Apple M1


**Minimum remaining energy versus time budget**, starting at rest, for three distances to the goal.

In [ ]:
for pos, label in [(N_POS - 50, "near"), (N_POS - 100, "medium"), (0, "far")]:
    metres_to_go = (N_POS - 1 - pos) * D_POS
    plt.plot(min_energy[0, pos, :], marker="o", label=f"{label}: {metres_to_go:.3f} m to go")
plt.xlabel("Time bins left (1 bin = 0.05 s)")
plt.ylabel("Minimum remaining energy [J]")
plt.title("Value function, starting at rest")
plt.legend()
plt.show()


## 5. Optimal trajectory

Starting at rest at position 0 with the full time budget, follow the optimal policy forward. `current_idx[k + 1]` is the current applied during step `k -> k + 1` (entry 0 is a placeholder).

In [ ]:
def extract_trajectory(best_current):
    """Roll the optimal policy forward from rest with the full time budget."""
    current_idx = np.zeros(N_POS, dtype=int)
    speed_idx = np.zeros(N_POS, dtype=int)
    t = N_TIME - 1
    for k in range(N_POS - 1):
        current_idx[k + 1] = best_current[speed_idx[k], k, t]
        speed_idx[k + 1] = NEXT_SPEED[speed_idx[k], current_idx[k + 1]]
        t -= STEP_DURATION[speed_idx[k], current_idx[k + 1]]
    return current_idx, speed_idx


In [ ]:
current_idx, speed_idx = extract_trajectory(best_current)
print(f"Minimum energy for the whole course: {min_energy[0, 0, N_TIME - 1]:.4f} J")


In [ ]:
plt.plot(current_idx * D_CURRENT, marker="o")
plt.xlabel("Position index (1 index = 5 mm)")
plt.ylabel("Applied current [A]")
plt.title("Optimal current along the course")
plt.show()


In [ ]:
plt.plot(speed_idx * D_SPEED * 3.6, marker="o")
plt.xlabel("Position index (1 index = 5 mm)")
plt.ylabel("Speed [km/h]")
plt.title("Optimal speed along the course")
plt.show()


In [ ]:
plt.plot(D_POS / (speed_idx[1:] * D_SPEED), marker="o")
plt.xlabel("Step index (1 index = 5 mm)")
plt.ylabel("Time per 5 mm step [s]")
plt.title("Time spent on each position step")
plt.show()
